# 3. Reward model

A second causal transformer reads, at position $t$, the item $i_t$ together with the quantised played ratio
$\rho_{t-1}$ of the previous item. Its state $z_t$ depends on $(i_{1:t}, \rho_{1:t-1})$: it has seen the candidate
item and the history, but not the response to the candidate. Four heads predict that response,

$$(\hat\rho_t,\ \hat c_t,\ \hat u_t,\ \hat q_t) = \sigma(W z_t),$$

the played ratio, the session continuation, the return, and the probability that the user chooses the item at
all. The loss is

$$\mathcal{L} = \operatorname{BCE}(\hat\rho, \rho) + \lambda_c \operatorname{BCE}(\hat c, c) + \lambda_u \operatorname{BCE}(\hat u, u) + \lambda_q\, \mathcal{L}_q,$$

with $\rho$ as a soft label and the retention terms taken over observed outcomes only. Logged events contain only
chosen items, so $\hat q$ is learned by contrast. One event per sequence is scored a second time with its item
replaced by the item $j$ of another event of the batch, so that replacements follow the empirical item
distribution $p$:

$$\mathcal{L}_q = -\tfrac{1}{2}\, \mathbb{E}\big[\log \hat q(i_t \mid h_t)\big] - \tfrac{1}{2}\, \mathbb{E}_{j \sim p}\big[\log\big(1 - \hat q(j \mid h_t)\big)\big],
\qquad \hat q^{*}(j \mid h) = \frac{P(j \mid h)}{P(j \mid h) + p(j)}.$$

Two copies, $a$ and $b$, are trained on disjoint halves of the users. Copy $a$ provides the rewards during
fine-tuning; copy $b$ only evaluates the resulting lists.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gptrec.data.batches import EventWindows
from gptrec.data.log import InteractionLog
from gptrec.evaluation.reward_model_metrics import (
    calibration_table,
    choice_quality,
    error_by_group,
    predict_heldout_feedback,
    retention_quality,
    reward_model_report,
)
from gptrec.models.backbone import TransformerConfig
from gptrec.paths import ARTIFACTS_DIR, PROCESSED_DATA_DIR
from gptrec.training.checkpoint import save_model
from gptrec.training.precision import default_device
from gptrec.training.supervised import OptimizationConfig, RewardModelTrainingConfig, train_reward_model

device = default_device()
log = InteractionLog.load(PROCESSED_DATA_DIR / "yambda-50m")

In [ ]:
MAX_POSITIONS = 128

config = RewardModelTrainingConfig(
    transformer=TransformerConfig(hidden_size=256, num_layers=4, num_heads=4, dropout=0.1, max_positions=MAX_POSITIONS),
    optimization=OptimizationConfig(epochs=5, batch_size=256, learning_rate=1e-3),
    window_stride=MAX_POSITIONS // 2,
)
all_users = np.arange(log.num_users)
user_folds = {"a": all_users[all_users % 2 == 0], "b": all_users[all_users % 2 == 1]}

reward_models, histories = {}, {}
for name, users in user_folds.items():
    reward_models[name], histories[name] = train_reward_model(log, config, device, users=users)
    save_model(reward_models[name], ARTIFACTS_DIR / f"reward_model_{name}.pt")

In [ ]:
pd.concat({name: pd.DataFrame(history).set_index("epoch") for name, history in histories.items()}, axis=1)

## Test period

Every test-period event is predicted from the events before it. The played ratio is compared with three reference
predictors: the training mean, the mean of the item, and the mean over the user's preceding events. The retention
signals are scored on events with observed outcomes, and $\hat q$ by its AUC for the logged item against a
replaced one.

In [ ]:
test_windows = EventWindows.for_heldout(log, "test", MAX_POSITIONS)
heldout = {name: predict_heldout_feedback(model, test_windows, batch_size=512, device=device) for name, model in reward_models.items()}

for name in reward_models:
    display(reward_model_report(heldout[name], log.items))
    display(retention_quality(heldout[name]))
    display(pd.Series(choice_quality(heldout[name])))

## Calibration

In [ ]:
figure, axes = plt.subplots(1, len(reward_models), figsize=(5 * len(reward_models), 4), squeeze=False)
for axis, name in zip(axes[0], reward_models):
    calibration = calibration_table(heldout[name].played_ratio, heldout[name].predicted_played_ratio)
    axis.plot(calibration.mean_predicted, calibration.mean_observed, marker="o")
    axis.plot([0, 1], [0, 1], linestyle="--", color="grey")
    axis.set(title=f"reward model {name}", xlabel="mean predicted played ratio", ylabel="mean observed played ratio")
figure.tight_layout()

## Error by item group

Mean signed error $\hat\rho - \rho$ by decile of item popularity and of item duration.

In [ ]:
def decile_of_item(values: np.ndarray) -> np.ndarray:
    return pd.qcut(pd.Series(values).rank(method="first"), 10, labels=False).to_numpy()


display(error_by_group(heldout["a"], decile_of_item(log.items.train_popularity)).rename_axis("popularity decile"))
display(error_by_group(heldout["a"], decile_of_item(log.items.duration_seconds)).rename_axis("duration decile"))

## Agreement of the two copies

Correlation between $\hat\rho^{\,a}$ and $\hat\rho^{\,b}$ on the same test events.

In [ ]:
np.corrcoef(heldout["a"].predicted_played_ratio, heldout["b"].predicted_played_ratio)[0, 1]